Analytical Model

Architecture DOR:

- p - physical channel (0 - wide bus; 1 - narrow bus)
- v - virtual channel (0 - read/write request, YX routing; 1 - read/write response, XY routing)
- s - source tile
- d - destination tile
- r - routing tile
- i - input direction
- o - output direction

Routing Tensor (RT):
RT[s][d][p][v][r][i][o]

Packet injection rate (f):
f[s][d][p][v] - packet injection rate for route s -> d via physical channel p, virtual channel v:
- f[s][d][0][0] - write request (s - perimeter; d - internal)
- f[s][d][0][1] - read response (s - internal; d - perimeter)
- f[s][d][1][0] - read request (s - perimeter; d - internal)
- f[s][d][1][1] - write response (s - internal; d - perimeter)

Packet Length (L):
L[p][v]

Router packet Arrival rate (A):
A[p][v][r][i][o] = Sum{any s; any d}(f[s][d][p][v] * RT[s][d][p][v][r][i][o])

Input buffer Arrival rate (IA):
IA[p][v][r][i] = Sum{any o}(A[p][v][r][i][o])

Output buffer Arrival rate (OA):
OA[p][v][r][o] = Sum{any i}(A[p][v][r][i][o])

Waiting Time of body flits from other directions in Input buffer (Tbody):
Tbody[p][v][r][i][o] = Sum{any k != i}(A[p][v][r][k][o] / L[p][v] * Sum{q from 1 to L[p][v]}(L - q)) = (L[p][v] - 1) / 2 * Sum{any k != i}(A[p][v][r][k][o])



- [0][0] - write request for horizontal (YX)
- [0][1] - read response for horizontal (XY)
- [0][2] - write request for vertical (XY)
- [0][3] - read response for vertical (YX)
- [1][0] - read request for horizontal (YX)
- [1][1] - write response for horizontal (XY)
- [1][2] - read request for vertical (XY)
- [1][3] - write response for vertical (YX)

- [0][0] -> [1][1]
- [0][2] -> [1][3]
- [1][0] -> [0][1]
- [1][2] -> [0][3]

In [1]:
import numpy as np

In [2]:
ROUTING = "MDOR"
PHYS_NUM = 2
VIRT_NUM = 2 if ROUTING == "DOR" else 4

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM

In [3]:
class Coord:
    def __init__(self, id=0):
        self.x = id % X_NUM
        self.y = id // Y_NUM

    def __repr__(self):
        return f"Coordinate(x={self.x}, y={self.y})"

    def is_perimeter(self):
        return (self.x == 0) or (self.x == X_NUM - 1) or (self.y == 0) or (self.y == Y_NUM - 1)

    def is_vertical(self):
        return (self.x == 0) or (self.x == X_NUM - 1)

    def is_horizontal(self):
        return self.is_perimeter() and not self.is_vertical()

    def id(self):
        return self.y * Y_NUM + self.x

In [4]:
# Returns next tile for given current and destination ones
# Algorith DOR
def next_tile_gen_yx(current_tile, dst_tile):
    next = Coord(current_tile.id())
    if current_tile.y < dst_tile.y:
        next.y += 1
    elif current_tile.y > dst_tile.y:
        next.y -= 1
    elif current_tile.x < dst_tile.x:
        next.x += 1
    elif current_tile.x > dst_tile.x:
        next.x -= 1
    return next

def next_tile_gen_xy(current_tile, dst_tile):
    next = Coord(current_tile.id())
    if current_tile.x < dst_tile.x:
        next.x += 1
    elif current_tile.x > dst_tile.x:
        next.x -= 1
    elif current_tile.y < dst_tile.y:
        next.y += 1
    elif current_tile.y > dst_tile.y:
        next.y -= 1
    return next

In [5]:
# Returns directions current_tile output ID and next_tile input ID
def encode_dirs(current_tile, next_tile):
    if current_tile.y < next_tile.y:
        return (SOUTH_IDX, NORTH_IDX)
    elif current_tile.y > next_tile.y:
        return (NORTH_IDX, SOUTH_IDX)
    elif current_tile.x < next_tile.x:
        return (EAST_IDX, WEST_IDX)
    elif current_tile.x > next_tile.x:
        return (WEST_IDX, EAST_IDX)
    return (LOCAL_IDX, LOCAL_IDX)

In [6]:
def calc_collision_possibility_internal(request_possibility, i, o, c, num):
    # The four other indices besides i
    others = [x for x in range(num) if x != i]
    others_num = num - 1
    others_comb_num = 2**others_num
    total = 0.0
    # Loop over all 16 combinations of (k1, k2, k3, k4) in {0,1}^4
    for mask in range(others_comb_num):  # from 0 to 15
        # Count how many bits are 1
        # and build the product F[...]^(k_t) * (1-F[...])^(1-k_t)
        ksum = 0
        prob_product = 1.0
        for bit_idx in range(others_num):
            # k_t is either 0 or 1
            k_t = (mask >> bit_idx) & 1
            p = request_possibility[others[bit_idx]][o]
            if k_t == 1:
                prob_product *= p
                ksum += 1
            else:
                prob_product *= (1 - p)

        # If k_1 + k_2 + k_3 + k_4 = c, add to sum
        if ksum == c:
            total += prob_product
    return total


def calc_collision_possibility(request_possibility, i, o, num):
    '''
    request_possibilty[i][o]
    -> collision_possibility
    '''
    collision_possibility = 0.0
    for c in range(1, num):
        collision_possibility_internal = calc_collision_possibility_internal(
            request_possibility, i, o, c, num)
        collision_possibility += collision_possibility_internal * \
            (c / (c + 1))
    return collision_possibility


def solve_request_possibility(payload_tensor, flow_control_possibility, max_iter=1000, tol=1e-8):
    '''
    payload_tensor[i][o]
    flow_control_possibility[i][o]
    -> request_possibility[i][o]
    '''
    request_possibility = payload_tensor.copy()

    (input_num, output_num) = payload_tensor.shape

    for _ in range(max_iter):
        request_possibility_old = request_possibility.copy()

        for i in range(input_num):
            for o in range(output_num):
                collision_possibility = calc_collision_possibility(
                    request_possibility, i, o, input_num)
                blocking_possibility = 1.0 - (1.0 - collision_possibility) * (1.0 - flow_control_possibility[i][o])

                denom = 1.0 - blocking_possibility
                if abs(denom) < 1e-14:
                    # Avoid dividing by zero.
                    # Could set F[i][j] to some fallback value.
                    request_possibility[i][o] = 0.999999 if denom < 0 else 0.0
                else:
                    request_possibility[i][o] = payload_tensor[i][o] / denom

        # Check for convergence
        diff = np.linalg.norm(request_possibility - request_possibility_old)
        if diff < tol:
            break

    # print(f"Finished in {it+1} iterations with diff={diff}")
    return request_possibility


def calc_blocking_possibility(request_possibility, flow_control_possibility):
    '''
        request_possibility[i][o]
        flow_control_possibility[i][o]
        -> blocking_possibility[i][o]
    '''
    blocking_possibility = request_possibility.copy()
    (input_num, output_num) = blocking_possibility.shape

    for i in range(input_num):
        for o in range(output_num):
            collision_possibility = calc_collision_possibility(request_possibility, i, o, input_num)
            blocking_possibility[i][o] = 1.0 - (1.0 - collision_possibility) * (1.0 - flow_control_possibility[i][o])

    return blocking_possibility

In [7]:
def is_traffic_exist(s, d, is_forward, is_vertical_master):
    s_coord = Coord(s)
    d_coord = Coord(d)
    if ROUTING == "DOR":
        if is_forward and (s_coord.is_perimeter() and not d_coord.is_perimeter()):
            return True
        if not is_forward and (not s_coord.is_perimeter() and d_coord.is_perimeter()):
            return True
        return False
    else:
        if is_forward and not is_vertical_master and (s_coord.is_horizontal() and not d_coord.is_perimeter()):
            return True
        if is_forward and is_vertical_master and (s_coord.is_vertical() and not d_coord.is_perimeter()):
            return True
        if not is_forward and not is_vertical_master and (not s_coord.is_perimeter() and d_coord.is_horizontal()):
            return True
        if not is_forward and is_vertical_master and (not s_coord.is_perimeter() and d_coord.is_vertical()):
            return True
        return False

def next_tile_gen_routing(current_tile, d_coord, is_forward, is_vertical_master):
    if ROUTING == "DOR":
        if is_forward:
            return next_tile_gen_yx(current_tile, d_coord)
        else:
            return next_tile_gen_xy(current_tile, d_coord)
    else:
        if is_forward and is_vertical_master:
            return next_tile_gen_xy(current_tile, d_coord)
        elif is_forward and not is_vertical_master:
            return next_tile_gen_yx(current_tile, d_coord)
        elif not is_forward and is_vertical_master:
            return next_tile_gen_yx(current_tile, d_coord)
        else:
            return next_tile_gen_xy(current_tile, d_coord)

def src_dst_routing_tensor(s, d, is_forward, is_vertical_master):
    '''
    -> routing_tensor[r][i][o]
    '''
    #              [r]       [i]      [o]
    rt = np.zeros((TILE_NUM, DIR_NUM, DIR_NUM))

    if not is_traffic_exist(s, d, is_forward, is_vertical_master):
        return rt

    s_coord = Coord(s)
    d_coord = Coord(d)
    current_tile = s_coord
    i_cur_dir = LOCAL_IDX
    while current_tile.id() != d:
        next_tile = next_tile_gen_routing(current_tile, d_coord, is_forward, is_vertical_master)
        (o_cur_dir, i_nxt_dir) = encode_dirs(current_tile, next_tile)
        rt[current_tile.id()][i_cur_dir][o_cur_dir] = 1.0
        current_tile = Coord(next_tile.id())
        i_cur_dir = i_nxt_dir
    rt[d][i_cur_dir][LOCAL_IDX] = 1.0

    return rt

def calc_routing_tensor():
    '''
    -> routing_tensor[p][v][s][d][r][i][o]
    '''
    routing_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM, TILE_NUM, DIR_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for s in range(TILE_NUM):
                for d in range(TILE_NUM):
                    is_forward = (v % 2 == 0)
                    is_vertical_master = (v > 1)
                    routing_tensor[p][v][s][d] = src_dst_routing_tensor(s, d, is_forward, is_vertical_master)
    return routing_tensor

In [8]:
#                                [p]       [v]
packet_length_tensor = np.zeros((PHYS_NUM, VIRT_NUM))

# Write request
packet_length_tensor[0][0] = 1.0
# Read response
packet_length_tensor[0][1] = 1.0
# Read request
packet_length_tensor[1][0] = 1.0
# Write response
packet_length_tensor[1][1] = 1.0

In [9]:
def calc_packet_injection_rate_tensor(pir):
    '''
    pir
    -> packet_injection_rate_tensor[p][v][s][d]
    '''
    packet_injection_rate_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for s in range(TILE_NUM):
                for d in range(TILE_NUM):
                    is_forward = (v % 2 == 0)
                    is_vertical_master = (v > 1)
                    if is_traffic_exist(s, d, is_forward, is_vertical_master):
                        packet_injection_rate_tensor[p][v][s][d] = pir

    return packet_injection_rate_tensor

In [10]:
def calc_router_packet_payload_tensor(routing_tensor, packet_injection_rate_tensor):
    '''
    routing_tensor[p][v][s][d][r][i][o]
    packet_injection_rate_tensor[p][v][s][d]
    -> router_packet_payload_tensor[p][v][r][i][o]
    '''
    multiplied = routing_tensor * packet_injection_rate_tensor[..., np.newaxis, np.newaxis, np.newaxis]
    router_packet_payload_tensor = multiplied.sum(axis=(2, 3))
    # [p] [v] [r] [i] [o]
    return router_packet_payload_tensor

In [11]:
def calc_router_input_packet_payload_tensor(router_packet_payload_tensor):
    '''
    router_packet_payload_tensor[p][v][r][i][o]
    -> router_input_packet_payload_tensor[p][v][r][i]
    '''
    # [p] [v] [r] [i]
    return router_packet_payload_tensor.sum(axis=-1)

In [12]:
def calc_router_output_packet_payload_tensor(router_packet_payload_tensor):
    '''
    router_packet_payload_tensor[p][v][r][i][o]
    -> router_output_packet_payload_tensor[p][v][r][o]
    '''
    # [p] [v] [r] [o]
    return router_packet_payload_tensor.sum(axis=3)

In [13]:
def calc_input_blocking_possibility_tensor(router_packet_payload_tensor, router_input_flow_control_possibility_tensor):
    '''
    router_packet_payload_tensor[p][v][r][i][o]
    router_input_flow_control_possibility_tensor[p][v][r][o]
    -> input_blocking_possibility_tensor[p][v][r][i][o]
    '''
    #                                             [p]       [v]       [r]       [i]      [o]
    input_blocking_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                request_possibility_solution = solve_request_possibility(
                    router_packet_payload_tensor[p][v][r], router_input_flow_control_possibility_tensor[p][v][r], max_iter=500, tol=1e-10)
                input_blocking_possibility_tensor[p][v][r] = calc_blocking_possibility(
                    request_possibility_solution, router_input_flow_control_possibility_tensor[p][v][r])

    return input_blocking_possibility_tensor

def calc_output_blocking_possibility_tensor(router_packet_payload_tensor, router_output_flow_control_possibility_tensor):
    '''
    router_packet_payload_tensor[p][v][r][i][o]
    router_output_flow_control_possibility_tensor[p][v][r][o]
    -> output_blocking_possibility_tensor[p][v][r][o]
    '''
    output_blocking_possibility_tensor_reshape = np.zeros((PHYS_NUM, TILE_NUM, VIRT_NUM, DIR_NUM))
    # [p][v][r][o]
    router_output_packet_payload_tensor = calc_router_output_packet_payload_tensor(router_packet_payload_tensor)
    # [p][r][v][o]
    router_output_packet_payload_tensor_reshape = np.zeros((PHYS_NUM, TILE_NUM, VIRT_NUM, DIR_NUM))
    router_output_flow_control_possibility_tensor_reshape = np.zeros((PHYS_NUM, TILE_NUM, VIRT_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for o in range(DIR_NUM):
                    router_output_packet_payload_tensor_reshape[p][r][v][o] = router_output_packet_payload_tensor[p][v][r][o]
                    router_output_flow_control_possibility_tensor_reshape[p][r][v][o] = router_output_flow_control_possibility_tensor[p][v][r][o]
    for p in range(PHYS_NUM):
        for r in range(TILE_NUM):
            request_possibility_solution = solve_request_possibility(
                router_output_packet_payload_tensor_reshape[p][r], router_output_flow_control_possibility_tensor_reshape[p][r], max_iter=500, tol=1e-10)
            output_blocking_possibility_tensor_reshape[p][r] = calc_blocking_possibility(
                request_possibility_solution, router_output_flow_control_possibility_tensor_reshape[p][r])
    output_blocking_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for o in range(DIR_NUM):
                    output_blocking_possibility_tensor[p][v][r][o] = output_blocking_possibility_tensor_reshape[p][r][v][o]
    return output_blocking_possibility_tensor

In [14]:
def calc_input_blocking_time_tensor(input_blocking_possibility_tensor):
    '''
    input_blocking_possibility_tensor[p][v][r][i][o]
    -> input_blocking_time_tensor[p][v][r][i][o]
    '''
    input_blocking_time_tensor = input_blocking_possibility_tensor.copy()

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for i in range(DIR_NUM):
                    for o in range(DIR_NUM):
                        input_blocking_time_tensor[p][v][r][i][o] /= (1 - input_blocking_time_tensor[p][v][r][i][o])

    return input_blocking_time_tensor

def calc_output_blocking_time_tensor(output_blocking_possibility_tensor):
    '''
    output_blocking_possibility_tensor[p][v][r][o]
    -> output_blocking_time_tensor[p][v][r][o]
    '''
    output_blocking_time_tensor = output_blocking_possibility_tensor.copy()

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for o in range(DIR_NUM):
                    output_blocking_time_tensor[p][v][r][o] /= (1 - output_blocking_time_tensor[p][v][r][o])
    return output_blocking_time_tensor

In [15]:
def calc_request_handle_time_tensor(blocking_time_tensor):
    '''
    blocking_time_tensor[p][v][r][i][o]
    -> request_handle_time_tensor[p][v][r][i][o]
    '''
    return blocking_time_tensor.copy() + 1.0

In [16]:
def calc_mean_input_handle_time_tensor(router_packet_payload_tensor, request_handle_time_tensor):
    '''
    router_packet_payload_tensor[p][v][r][i][o]
    request_handle_time_tensor[p][v][r][i][o]
    -> mean_input_handle_time_tensor[p][v][r][i]
    '''
    mean_input_handle_time_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
    router_input_packet_payload_tensor = calc_router_input_packet_payload_tensor(router_packet_payload_tensor)

    handle_time_tensor = router_packet_payload_tensor * request_handle_time_tensor
    sum_input_handle_time_tensor = handle_time_tensor.sum(axis=-1)

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for i in range(DIR_NUM):
                    mean_input_handle_time_tensor[p][v][r][i] = 0. if (abs(sum_input_handle_time_tensor[p][v][r][i]) < 1e-9) else sum_input_handle_time_tensor[p][v][r][i] / \
                        router_input_packet_payload_tensor[p][v][r][i]

    return mean_input_handle_time_tensor

def calc_mean_output_handle_time_tensor(router_output_packet_payload_tensor, request_handle_time_tensor):
    '''
    router_output_packet_payload_tensor[p][v][r][o]
    request_handle_time_tensor[p][v][r][o]
    -> mean_output_handle_time_tensor[p][v][r][o]
    '''
    return router_output_packet_payload_tensor * request_handle_time_tensor

In [17]:
def mm1n_queue(lambda_arrival, E_T, N):
    '''
    `lambda_arrival` -- intensivity of requests
    `E_T` -- average request handle time
    `N` -- number of queue entries
    -> (mean_depth, p[N]) -- mean queue depth and probability of N requests in the queue
    '''
    utilization = lambda_arrival * E_T
    pk_values = [(1 - utilization) * utilization**k / (1 - utilization**(N+1))
                 for k in range(N+1)]
    mean_depth = sum((k - 1) * pk_values[k] for k in range(1, N+1))
    return (mean_depth, pk_values[N])

In [18]:
def calc_mean_input_queue_depth_tensor(router_input_packet_payload_tensor, mean_input_handle_time_tensor):
    '''
    router_input_packet_payload_tensor[p][v][r][i]
    mean_input_handle_time_tensor[p][v][r][i]
    -> (mean_input_queue_depth_tensor[p][v][r][i], input_queue_full_probability_tensor[p][v][r][i])
    '''
    mean_input_queue_depth_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
    input_queue_full_probability_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for i in range(DIR_NUM):
                    (mean_input_queue_depth_tensor[p][v][r][i], input_queue_full_probability_tensor[p][v][r][i]) = mm1n_queue(
                        router_input_packet_payload_tensor[p][v][r][i], mean_input_handle_time_tensor[p][v][r][i], 2)
    return (mean_input_queue_depth_tensor, input_queue_full_probability_tensor)

def calc_mean_output_queue_depth_tensor(router_output_packet_payload_tensor, mean_output_handle_time_tensor):
    '''
    router_output_packet_payload_tensor[p][v][r][o]
    mean_output_handle_time_tensor[p][v][r][o]
    -> (mean_output_queue_depth_tensor[p][v][r][o], output_queue_full_probability_tensor[p][v][r][o])
    '''
    mean_output_queue_depth_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
    output_queue_full_probability_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for o in range(DIR_NUM):
                    (mean_output_queue_depth_tensor[p][v][r][o], output_queue_full_probability_tensor[p][v][r][o]) = mm1n_queue(
                        router_output_packet_payload_tensor[p][v][r][o], mean_output_handle_time_tensor[p][v][r][o], 2)
    return (mean_output_queue_depth_tensor, output_queue_full_probability_tensor)

In [19]:
def calc_input_average_latency(routing_tensor, mean_input_queue_depth_tensor, mean_input_handle_time_tensor):
    '''
    routing_tensor[r][i][o]
    mean_input_queue_depth_tensor[r][i]
    mean_input_handle_time_tensor[r][i]
    -> input_average_latency
    '''
    input_routing_tensor = routing_tensor.sum(axis=-1)

    input_average_latency = 0.0
    for r in range(TILE_NUM):
        for i in range(DIR_NUM):
            if abs(input_routing_tensor[r][i]) > 1e-9:
                input_average_latency += (mean_input_queue_depth_tensor[r][i] + 1.) * \
                    mean_input_handle_time_tensor[r][i]
    return input_average_latency

def calc_output_average_latency(routing_tensor, mean_output_queue_depth_tensor, mean_output_handle_time_tensor):
    '''
    routing_tensor[r][i][o]
    mean_output_queue_depth_tensor[r][o]
    mean_output_handle_time_tensor[r][o]
    -> output_average_latency
    '''
    output_routing_tensor = routing_tensor.sum(axis=1)

    output_average_latency = 0.0
    for r in range(TILE_NUM):
        for o in range(DIR_NUM):
            if abs(output_routing_tensor[r][o]) > 1e-9:
                output_average_latency += (mean_output_queue_depth_tensor[r][o] + 1.) * \
                    mean_output_handle_time_tensor[r][o]
    return output_average_latency

In [20]:
def calc_average_latency_tensor(routing_tensor, mean_input_queue_depth_tensor, mean_input_handle_time_tensor, mean_output_queue_depth_tensor, mean_output_handle_time_tensor):
    '''
    routing_tensor[p][v][s][d][r][i][o]
    mean_input_queue_depth_tensor[p][v][r][i]
    mean_input_handle_time_tensor[p][v][r][i]
    mean_output_queue_depth_tensor[p][v][r][o]
    mean_output_handle_time_tensor[p][v][r][o]
    -> average_latency_tensor[p][v][s][d]
    '''
    average_latency_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for s in range(TILE_NUM):
                for d in range(TILE_NUM):
                    if abs(routing_tensor[p][v][s][d].sum()) > 1e-9:
                        input_average_latency = calc_input_average_latency(
                            routing_tensor[p][v][s][d], mean_input_queue_depth_tensor[p][v], mean_input_handle_time_tensor[p][v])
                        output_average_latency = calc_output_average_latency(
                            routing_tensor[p][v][s][d], mean_output_queue_depth_tensor[p][v], mean_output_handle_time_tensor[p][v])
                        average_latency_tensor[p][v][s][d] = input_average_latency + output_average_latency

                # print(Coord(s), Coord(d), average_latency[s][d])

    return average_latency_tensor

In [21]:
def update_input_flow_control_possibility_tensor(output_queue_full_probability_tensor):
    '''
    output_queue_full_probability_tensor[p][v][r][o]
    -> router_input_flow_control_possibility_tensor[p][v][r][i][o]
    '''
    expanded = np.expand_dims(output_queue_full_probability_tensor, axis=3)
    return np.repeat(expanded, DIR_NUM, axis=3)

def update_output_flow_control_possibility_tensor(input_queue_full_probability_tensor):
    '''
    input_queue_full_probability_tensor[p][v][r][i]
    -> router_output_flow_control_possibility_tensor[p][v][r][o]
    '''
    output_flow_control_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                r_coord = Coord(r)
                if r_coord.y > 0:
                    r_coord_north = Coord(r)
                    r_coord_north.y -= 1
                    output_flow_control_possibility_tensor[p][v][r][NORTH_IDX] = input_queue_full_probability_tensor[p][v][r_coord_north.id()][SOUTH_IDX]
                if r_coord.y < Y_NUM - 1:
                    r_coord_south = Coord(r)
                    r_coord_south.y += 1
                    output_flow_control_possibility_tensor[p][v][r][SOUTH_IDX] = input_queue_full_probability_tensor[p][v][r_coord_south.id()][NORTH_IDX]
                if r_coord.x > 0:
                    r_coord_west = Coord(r)
                    r_coord_west.x -= 1
                    output_flow_control_possibility_tensor[p][v][r][WEST_IDX] = input_queue_full_probability_tensor[p][v][r_coord_west.id()][EAST_IDX]
                if r_coord.x < X_NUM - 1:
                    r_coord_east = Coord(r)
                    r_coord_east.x += 1
                    output_flow_control_possibility_tensor[p][v][r][EAST_IDX] = input_queue_full_probability_tensor[p][v][r_coord_east.id()][WEST_IDX]
                if not r_coord.is_perimeter():
                    p_inp = (p + 1) % PHYS_NUM
                    v_inp = (v + 1) % VIRT_NUM
                    output_flow_control_possibility_tensor[p][v][r][LOCAL_IDX] = input_queue_full_probability_tensor[p_inp][v_inp][r][LOCAL_IDX]
    return output_flow_control_possibility_tensor

In [22]:
def calc_average_latency_pir_pipeline(pir, max_iter=10, tol=1e-9):
  pir_per_mem_tile = pir / (X_NUM - 2) / (Y_NUM - 2)
  routing_tensor = calc_routing_tensor()
  #                                                        [p]       [v]       [r]       [i]      [o]
  router_input_flow_control_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM, DIR_NUM))
  #                                                         [p]       [v]       [r]       [o]
  router_output_flow_control_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
  packet_injection_rate_tensor = calc_packet_injection_rate_tensor(pir_per_mem_tile)
  router_packet_payload_tensor = calc_router_packet_payload_tensor(routing_tensor, packet_injection_rate_tensor)
  router_input_packet_payload_tensor = calc_router_input_packet_payload_tensor(router_packet_payload_tensor)
  router_output_packet_payload_tensor = calc_router_output_packet_payload_tensor(router_packet_payload_tensor)
  for it in range(max_iter):
    input_blocking_possibility_tensor = calc_input_blocking_possibility_tensor(router_packet_payload_tensor, router_input_flow_control_possibility_tensor)
    input_blocking_time_tensor = calc_input_blocking_time_tensor(input_blocking_possibility_tensor)
    input_request_handle_time_tensor = calc_request_handle_time_tensor(input_blocking_time_tensor)
    mean_input_handle_time_tensor = calc_mean_input_handle_time_tensor(router_packet_payload_tensor, input_request_handle_time_tensor)
    (mean_input_queue_depth_tensor, input_queue_full_probability_tensor) = calc_mean_input_queue_depth_tensor(router_input_packet_payload_tensor, mean_input_handle_time_tensor)
    output_blocking_possibility_tensor = calc_output_blocking_possibility_tensor(router_packet_payload_tensor, router_output_flow_control_possibility_tensor)
    output_blocking_time_tensor = calc_output_blocking_time_tensor(output_blocking_possibility_tensor)
    output_request_handle_time_tensor = calc_request_handle_time_tensor(output_blocking_time_tensor)
    mean_output_handle_time_tensor = output_request_handle_time_tensor.copy()
    (mean_output_queue_depth_tensor, output_queue_full_probability_tensor) = calc_mean_output_queue_depth_tensor(router_output_packet_payload_tensor, mean_output_handle_time_tensor)

    router_input_flow_control_possibility_tensor_old = router_input_flow_control_possibility_tensor.copy()
    router_output_flow_control_possibility_tensor_old = router_output_flow_control_possibility_tensor.copy()
    router_input_flow_control_possibility_tensor = update_input_flow_control_possibility_tensor(output_queue_full_probability_tensor)
    router_output_flow_control_possibility_tensor = update_output_flow_control_possibility_tensor(input_queue_full_probability_tensor)
    input_diff = np.linalg.norm(router_input_flow_control_possibility_tensor - router_input_flow_control_possibility_tensor_old)
    output_diff = np.linalg.norm(router_output_flow_control_possibility_tensor - router_output_flow_control_possibility_tensor_old)
    if input_diff < tol and output_diff < tol:
      break
  print(f"finished in iter={it}; input_diff={input_diff}; output_diff={output_diff}")
  average_latency_tensor = calc_average_latency_tensor(routing_tensor, mean_input_queue_depth_tensor, mean_input_handle_time_tensor, mean_output_queue_depth_tensor, mean_output_handle_time_tensor)
  return average_latency_tensor

In [23]:
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150]
# pir_list = [0.05, 0.1]
PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
for pir in pir_list:
  lat = calc_average_latency_pir_pipeline(pir)
  lat_wreq = lat[0][0].sum() / PAIR_NUM + 1
  lat_wresp = lat[1][1].sum() / PAIR_NUM + 1
  lat_rreq = lat[1][0].sum() / PAIR_NUM + 1
  lat_rresp = lat[0][1].sum() / PAIR_NUM + 1
  lat_write = lat_wreq + lat_wresp
  lat_read = lat_rreq + lat_rresp
  test_wreq = lat[0][0][0][11] + 1
  test_wresp = lat[1][1][11][0] + 1
  test_write = test_wreq + test_wresp
  print(f"{pir} done")
  print(f"latency_wreq = {lat_wreq}; latency_wresp = {lat_wresp}; latency_write = {lat_write}")
  print(f"latency_rreq = {lat_rreq}; latency_rresp = {lat_rresp}; latency_read = {lat_read}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")

finished in iter=4; input_diff=8.342765655728936e-11; output_diff=3.564178909630472e-11
0.025 done
latency_wreq = 17.821128301628413; latency_wresp = 17.826889281112177; latency_write = 35.64801758274059
latency_rreq = 17.821128301628413; latency_rresp = 17.826889281112177; latency_read = 35.64801758274059
latency_write [0][0] -> [1][1]: wreq = 7.0237536903107305; wresp = 7.013545124333856; write = 14.037298814644586
finished in iter=6; input_diff=3.24072246124511e-11; output_diff=1.3645358846543129e-11
0.05 done
latency_wreq = 18.056747532419266; latency_wresp = 18.068891442473824; latency_write = 36.12563897489309
latency_rreq = 18.056747532419266; latency_rresp = 18.068891442473824; latency_read = 36.12563897489309
latency_write [0][0] -> [1][1]: wreq = 7.066136018785864; wresp = 7.041000913243114; write = 14.107136932028979
finished in iter=7; input_diff=4.406126175846714e-10; output_diff=1.3323414866613564e-10
0.075 done
latency_wreq = 18.38350073496853; latency_wresp = 18.4027441

In [23]:
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150]
# pir_list = [0.05, 0.1]
PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
for pir in pir_list:
  lat = calc_average_latency_pir_pipeline(pir)
  lat_wreq = (lat[0][0].sum() + lat[0][2].sum()) / PAIR_NUM + 1
  lat_wresp = (lat[1][1].sum() + lat[1][3].sum()) / PAIR_NUM + 1
  lat_rreq = (lat[1][0].sum() + lat[1][2].sum()) / PAIR_NUM + 1
  lat_rresp = (lat[0][1].sum() + lat[0][3].sum()) / PAIR_NUM + 1
  lat_write = lat_wreq + lat_wresp
  lat_read = lat_rreq + lat_rresp
  test_wreq = lat[0][2][0][11] + 1
  test_wresp = lat[1][3][11][0] + 1
  test_write = test_wreq + test_wresp
  print(f"{pir} done")
  print(f"latency_wreq = {lat_wreq}; latency_wresp = {lat_wresp}; latency_write = {lat_write}")
  print(f"latency_rreq = {lat_rreq}; latency_rresp = {lat_rresp}; latency_read = {lat_read}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")

finished in iter=3; input_diff=1.7293352847713137e-11; output_diff=8.199424182793832e-12
0.025 done
latency_wreq = 17.79349265493556; latency_wresp = 17.79955504455451; latency_write = 35.59304769949007
latency_rreq = 17.79349265493556; latency_rresp = 17.79955504455451; latency_read = 35.59304769949007
latency_write [0][0] -> [1][1]: wreq = 7.022723192680962; wresp = 7.0228271435441645; write = 14.045550336225126
finished in iter=4; input_diff=1.5509454942272896e-11; output_diff=3.895145173743581e-12
0.05 done
latency_wreq = 17.94439541081482; latency_wresp = 17.957713172230537; latency_write = 35.90210858304536
latency_rreq = 17.94439541081482; latency_rresp = 17.957713172230537; latency_read = 35.90210858304536
latency_write [0][0] -> [1][1]: wreq = 7.05337879695727; wresp = 7.053255802835099; write = 14.106634599792368
finished in iter=4; input_diff=7.969018063073763e-10; output_diff=2.023218825391026e-10
0.075 done
latency_wreq = 18.120441018741285; latency_wresp = 18.142234299628